# Incremental Updates: Monitor New Filings

Goal: for an ETL or monitoring workflow, find banks that have filed (or amended their filing) since your last check.

**Use case:**
- Nightly cron job that pulls only new/amended filings
- Regulatory dashboard that refreshes as institutions file
- Audit trail of filing timestamps

The FFIEC REST API exposes two endpoints that support this pattern:
- `collect_filers_since_date` — just the RSSD IDs that filed after a given date
- `collect_filers_submission_date_time` — RSSD IDs **plus** the exact submission timestamp

## Setup

In [ ]:
import os
from datetime import datetime, timedelta
import pandas as pd
from ffiec_data_connect import (
    OAuth2Credentials,
    collect_data,
    collect_filers_since_date,
    collect_filers_submission_date_time,
)

creds = OAuth2Credentials(
    username=os.environ['FFIEC_USERNAME'],
    bearer_token=os.environ['FFIEC_BEARER_TOKEN'],
)

## Pattern 1: Which RSSD IDs filed since last run?

In a real ETL job, you'd persist `last_run_date` to a database or file between runs. Here we'll simulate it as 30 days ago.

In [ ]:
# Your current reporting period (the quarter you're pulling)
REPORTING_PERIOD = '12/31/2024'

# Last time your ETL ran
last_run = (datetime.now() - timedelta(days=30)).strftime('%m/%d/%Y')
print(f'Looking for filings since: {last_run}')

new_filers = collect_filers_since_date(
    creds,
    reporting_period=REPORTING_PERIOD,
    since_date=last_run,
)

print(f'{len(new_filers)} institutions filed since {last_run}')
print(f'First 5: {new_filers[:5]}')

## Pattern 2: Get the submission timestamps

When you need to know **exactly when** each institution filed (for SLA monitoring, audit trails, or ordering by recency).

In [ ]:
submissions = collect_filers_submission_date_time(
    creds,
    since_date=last_run,
    reporting_period=REPORTING_PERIOD,
    output_type='pandas',
)

print(f'Got {len(submissions)} submissions')
submissions.head()

## Find the most recent filers

In [ ]:
# Sort by submission time, newest first
submissions['datetime'] = pd.to_datetime(submissions['datetime'])
recent = submissions.sort_values('datetime', ascending=False).head(20)

print('Most recent filings:')
recent[['rssd', 'datetime']]

## ETL pattern: fetch data only for new filers

Instead of re-downloading every bank's XBRL every night, only pull the ones that filed or amended since your last run.

In [ ]:
# Simulated ETL loop — in production you'd persist results to a database
updated_records = []

# Take first 5 for the example; in production you'd iterate all new_filers
for rssd in new_filers[:5]:
    print(f'Fetching {rssd}...', end=' ')
    try:
        df = collect_data(
            creds,
            reporting_period=REPORTING_PERIOD,
            rssd_id=rssd,
            series='call',
            output_type='pandas',
        )
        df['rssd'] = rssd
        updated_records.append(df)
        print(f'OK ({len(df)} items)')
    except Exception as e:
        print(f'FAIL: {e}')

updated = pd.concat(updated_records, ignore_index=True) if updated_records else pd.DataFrame()
print(f'\nUpdated {len(updated_records)} institutions, {len(updated):,} total data points')

## Production checklist

For a real scheduled job:

1. **Persist `last_run_date`** — store it in your database or a small state file
2. **Handle failures** — if one bank errors, don't fail the whole run (wrap in try/except, log failures)
3. **Idempotency** — upsert by `(rssd, reporting_period, mdrm)` so re-runs are safe
4. **Token rotation** — JWT tokens expire every 90 days; `creds.is_expired` is your friend
5. **Rate limiting** — built-in limiter handles the 2500/hr cap, but for very large panels consider async + batching
6. **Monitoring** — log counts of fetched banks, record durations, alert on anomalies